# Encoder B (independent judge), room measurement, decision thresholds

PHC-351 · Hiding a Voice · Bhavishay (24324008), IIT Roorkee

Code files 06_model_B.py, 07_rooms.py, 08_thresholds.py from `code/`, as Colab cells in run order.

**How to run:** Runtime → Change runtime type → **T4 GPU** → Save, then **Runtime → Run all**. Allow Google Drive access when asked.

Needs the project folder on your Google Drive (the sweep recordings). **06 trains encoder B (5 epochs): the long part.** 07 rebuilds the four rooms from the sweep recordings and checks them against the saved rooms; 08 computes the thresholds with the B trained in 06.

**Changes from the original cells, so that the notebook runs on its own:** a setup cell at the top; cells marked `ADDED for the re-run` (names the original session defined in cells that were not kept, and two result printouts); and file paths, marked `# re-run:`, so that everything is written into a run folder in this Colab session and the files on Drive are never overwritten. Everything else is the code as run; `code/` keeps the original paths.

## Setup (added)

In [ ]:
# ===== SETUP (added so this notebook runs top to bottom from GitHub; not part of the original runs) =====
import os, glob, shutil, torch, torchaudio, numpy as np, soundfile as sf
RUN = '/content/PHC351_rerun'          # this session's working folder; Colab deletes it when the session ends
os.makedirs(RUN, exist_ok=True)

# Audio files: newer torchaudio needs the extra package torchcodec for load/save.
# If it is not installed, load/save go through soundfile instead (same samples, same shapes).
try:
    torchaudio.save(f'{RUN}/_probe.wav', torch.zeros(1, 160), 16000); torchaudio.load(f'{RUN}/_probe.wav')
except Exception:
    def _load(path, *a, **k):
        x, sr = sf.read(path, dtype='float32', always_2d=True)
        return torch.from_numpy(np.ascontiguousarray(x.T)), sr
    def _save(path, x, sr, *a, **k):
        sf.write(path, np.ascontiguousarray(x.detach().cpu().numpy().T), sr, subtype='FLOAT')
    torchaudio.load, torchaudio.save = _load, _save
    print('audio load/save: soundfile (torchcodec not installed)')

NEED = ['code/modelA_speaker_encoder.pt', 'testing_data/recordings/sweep_*.wav', 'testing_data/recordings/clean_*.wav', 'testing_data/recordings/measured_rirs.pt', 'demo_best_config/A_original.wav', 'training_data/heldout_subset/*.flac']
# Inputs: read from the project folder on Google Drive and copied into RUN.
# The code below writes only into RUN, so the original files on Drive are never overwritten.
from google.colab import drive
drive.mount('/content/drive')
ORIG = '/content/drive/MyDrive/PHC351_24324008_VoiceProtection'
if not os.path.isdir(ORIG):
    raise SystemExit(f'Project folder not found on this Google Drive: {ORIG}\n'
                     'This notebook needs the recordings and models in that folder (the recordings are not in the repository).')
n = 0
for sub, pat in [('code', '*.pt'), ('testing_data/recordings', '*.wav'), ('testing_data/recordings', '*.pt'),
                 ('testing_data/results', 'thresholds_measured.pt'), ('training_data/heldout_subset', '*.flac'),
                 ('demo_best_config', '*.wav')]:
    os.makedirs(f'{RUN}/{sub}', exist_ok=True)
    for f in glob.glob(f'{ORIG}/{sub}/{pat}'):
        dst = f'{RUN}/{sub}/{os.path.basename(f)}'
        if not os.path.exists(dst): shutil.copy2(f, dst); n += 1
os.makedirs(f'{RUN}/figures', exist_ok=True)
print(f'copied {n} input files from the project folder into {RUN} (the project folder is only read)')
missing = [p for p in NEED if not glob.glob(f'{RUN}/{p}')]
print('all inputs found' if not missing else f'MISSING inputs (the cells that use them will fail): {missing}')

## 06_model_B.py

Encoder B, the independent judge, in a fresh session: restore the data and train (seed 1234).

In [ ]:
import os, glob, random, time, json, torch, torchaudio, torch.nn as nn, numpy as np
from torch.utils.data import Dataset, DataLoader
from google.colab import drive; drive.mount('/content/drive')

OUT = f'{RUN}/phc351'; os.makedirs(OUT, exist_ok=True)   # re-run: run folder (original: '/content/drive/MyDrive/phc351')
os.chdir('/content')
if not os.path.exists('/content/LibriSpeech/train-clean-100'):
    os.system("curl -sL -C - -o tc100.tar.gz "
              "https://www.openslr.org/resources/12/train-clean-100.tar.gz && tar -xzf tc100.tar.gz")

SR, N_FFT, HOP, WIN, N_MELS = 16000, 512, 160, 400, 80
CHUNK = 3*SR
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
_window = torch.hann_window(WIN)
_fb = torchaudio.functional.melscale_fbanks(N_FFT//2+1, 0.0, SR/2, N_MELS, SR,
                                            norm=None, mel_scale="htk")

def logmel(wav):
    X = torch.stft(wav, N_FFT, HOP, WIN, _window.to(wav.device),
                   center=True, return_complex=True)
    return torch.log(_fb.T.to(wav.device) @ (X.abs()**2) + 1e-6)

files = sorted(glob.glob('/content/LibriSpeech/train-clean-100/*/*/*.flac'))
pairs = [(p, p.split('/')[-3]) for p in files]
speakers = sorted({s for _, s in pairs}); random.Random(0).shuffle(speakers)
n_tr = int(0.8*len(speakers))
train_spk, held_spk = set(speakers[:n_tr]), set(speakers[n_tr:])
spk2idx = {s:i for i,s in enumerate(sorted(train_spk))}
print("files", len(files), "| train spk", len(train_spk), "| held", len(held_spk))

class SpeakerSet(Dataset):
    def __init__(self, pairs, keep, labelled=True):
        self.items = [(p,s) for p,s in pairs if s in keep]; self.labelled = labelled
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        path, spk = self.items[i]
        wav, _ = torchaudio.load(path); wav = wav.mean(0)
        if wav.shape[0] < CHUNK:
            wav = torch.nn.functional.pad(wav, (0, CHUNK-wav.shape[0]))
        else:
            st = random.randint(0, wav.shape[0]-CHUNK); wav = wav[st:st+CHUNK]
        return logmel(wav), (spk2idx[spk] if self.labelled else -1)

train_dl = DataLoader(SpeakerSet(pairs, train_spk), batch_size=32, shuffle=True,
                      num_workers=2, drop_last=True)

class SpeakerNetB(nn.Module):
    def __init__(self, n_classes, emb_dim=256):
        super().__init__()
        def blk(ci, co, k, s):
            return nn.Sequential(nn.Conv2d(ci, co, k, s, k//2),
                                 nn.BatchNorm2d(co), nn.ReLU(inplace=True))
        self.conv = nn.Sequential(blk(1,48,5,(2,1)), blk(48,96,3,(2,1)),
                                  blk(96,160,3,(2,2)), blk(160,160,3,(2,1)),
                                  blk(160,160,3,(1,1)))
        self.proj = nn.Sequential(nn.Linear(160*5*2,512), nn.ReLU(inplace=True),
                                  nn.Linear(512, emb_dim))
        self.bn = nn.BatchNorm1d(emb_dim); self.head = nn.Linear(emb_dim, n_classes)
    def forward(self, x):
        h = self.conv(x.unsqueeze(1)); B,C,F,T = h.shape
        h = h.reshape(B, C*F, T)
        e = self.bn(self.proj(torch.cat([h.mean(2),
              h.var(2, unbiased=False).clamp_min(1e-8).sqrt()], 1)))
        return e, self.head(e)

torch.manual_seed(1234); random.seed(1234)
modelB = SpeakerNetB(len(spk2idx)).to(dev)
opt = torch.optim.Adam(modelB.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss()
print("B params(M):", round(sum(p.numel() for p in modelB.parameters())/1e6,2))

for ep in range(5):
    modelB.train(); t0=time.time(); tot=cor=0; run=0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(dev), yb.to(dev)
        _, lg = modelB(xb); loss = lossf(lg, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        run += loss.item()*yb.size(0); cor += (lg.argmax(1)==yb).sum().item(); tot += yb.size(0)
    print(f"B ep {ep+1}/5 | loss {run/tot:.3f} | acc {100*cor/tot:.1f}% | {time.time()-t0:.0f}s", flush=True)
    torch.save(modelB.state_dict(), f'{OUT}/modelB.pt')

print("DONE — modelB.pt saved to Drive")

## 07_rooms.py

Room measurement: the sine sweep; trimming, alignment and RT60; the regularised deconvolution and the measurement loop.

In [ ]:
import torch, torchaudio, math, os
R = RUN   # re-run: run folder (original: '/content/drive/MyDrive/PHC351_24324008_VoiceProtection')
os.makedirs(f'{R}/testing_data/recordings', exist_ok=True)
SR=16000
T,f1,f2 = 10.0, 20.0, 8000.0
t = torch.linspace(0, T, int(SR*T))
K = 2*math.pi*f1*T/math.log(f2/f1)
sweep = torch.sin(K*(torch.exp(t/T*math.log(f2/f1)) - 1))
sweep = torch.cat([torch.zeros(SR), sweep*0.8, torch.zeros(2*SR)])   # 1s lead, 2s tail for reverb
torchaudio.save(f'{R}/testing_data/recordings/sweep_reference.wav', sweep.unsqueeze(0), SR)
print("saved sweep_reference.wav —", sweep.shape[0]/SR, "seconds")

In [ ]:
# ===== next cell =====
def load16k(path):
    x, sr = torchaudio.load(path); x = x.mean(0)
    if sr != SR: x = torchaudio.functional.resample(x, sr, SR)
    return x

# convert m4a -> 16 kHz mono wav via ffmpeg (turn 446)
def convert_all():
    for p in sorted(glob.glob(f'{REC}/*')):
        base = os.path.basename(p); stem, ext = os.path.splitext(base)
        if ext.lower() not in ('.m4a','.mp3','.mov','.mp4','.aac','.caf'): continue
        s = stem.lower().replace('pro_', 'prot_')
        if not re.match(r'(sweep|clean|prot)_', s): continue
        out = f'{REC}/{s}.wav'
        if os.path.exists(out): continue
        subprocess.run(['ffmpeg','-y','-v','error','-i',p,'-ac','1','-ar',str(SR),'-c:a','pcm_s16le', out])

# cut the recording to the part that contains the sweep (turn 454)
def trim_to_sweep(rec, ref, pad=SR):
    n = 1 << (rec.shape[0] + ref.shape[0] - 1).bit_length()
    c = torch.fft.irfft(torch.fft.rfft(rec, n) * torch.fft.rfft(ref.flip(0), n), n)
    lag = int(c[:rec.shape[0]].argmax()) - (ref.shape[0] - 1)
    a = max(0, lag - pad); b = min(rec.shape[0], max(0, lag) + ref.shape[0] + 2*pad)
    return rec[a:b], lag

# reverberation time: Schroeder backward integration, T20 x 3, truncated at the noise floor (turn 450)
def rt60(h):
    e  = h**2
    nf = e[-e.shape[0]//10:].mean()                       # noise floor
    m  = (e > 10*nf).nonzero()
    e  = e[:int(m[-1])+1] if len(m) else e
    s  = torch.flip(torch.cumsum(torch.flip(e,[0]),0),[0])
    db = 10*torch.log10(s/s[0] + 1e-12)
    try: i5, i25 = int((db<=-5).nonzero()[0]), int((db<=-25).nonzero()[0])
    except IndexError: return float('nan')
    return 3.0*(i25-i5)/SR

# line a re-recording up with the original: cross-correlate, crop, match RMS (turn 456)
def align_to(rec, ref):
    n = 1 << (rec.shape[0] + ref.shape[0] - 1).bit_length()
    c = torch.fft.irfft(torch.fft.rfft(rec, n) * torch.fft.rfft(ref.flip(0), n), n)
    lag = max(0, int(c[:rec.shape[0]].argmax()) - (ref.shape[0] - 1))
    y = rec[lag : lag + ref.shape[0]]
    y = torch.nn.functional.pad(y, (0, max(0, ref.shape[0] - y.shape[0])))
    return y * (ref.std() / y.std().clamp_min(1e-9))

In [ ]:
# ===== ADDED for the re-run: names that the original session defined in cells that were not kept =====
import re, subprocess, random, torch.nn as nn
REC = f'{R}/testing_data/recordings'
ready = [r for r in ['normalroom', 'bigroomcenter', 'bigroomcorner', 'stair']
         if os.path.exists(f'{REC}/sweep_{r}.wav') and os.path.exists(f'{REC}/clean_{r}.wav')]
HELD = ['stair']                                       # as stored in the saved measured_rirs.pt
ref = sweep[SR: SR + int(SR * T)]                      # the bare sweep; with it the processing reproduces the saved rooms exactly (overnight run v2, section C)
def fit(x, n=CHUNK): return torch.nn.functional.pad(x, (0, max(0, n - x.shape[0])))[:n]
def apply_rir(x, h):                                   # as in 09_main_run.py
    L = x.shape[0] + h.shape[0] - 1; nf = 1 << (L - 1).bit_length()
    y = torch.fft.irfft(torch.fft.rfft(x, nf) * torch.fft.rfft(h.to(x.device), nf), nf)[:x.shape[0]]
    return y / y.abs().max().clamp_min(1e-9) * x.abs().max()
def emb(net, x):                                       # as in 09_main_run.py
    with torch.no_grad(): e, _ = net(logmel(x.to(dev)).unsqueeze(0))
    return torch.nn.functional.normalize(e, dim=1)[0]
cos = lambda a, b: float((a * b).sum())
nrm = lambda v: torch.nn.functional.normalize(v, dim=0)

class SpeakerNet(nn.Module):
    def __init__(s, nc, d=192):
        super().__init__()
        b = lambda ci,co,st: nn.Sequential(nn.Conv2d(ci,co,3,st,1), nn.BatchNorm2d(co), nn.ReLU(True))
        s.conv = nn.Sequential(b(1,32,(2,1)), b(32,64,(2,1)), b(64,128,(2,1)), b(128,128,(2,1)))
        s.proj = nn.Linear(128*5*2, d); s.bn = nn.BatchNorm1d(d); s.head = nn.Linear(d, nc)
    def forward(s, x):
        h = s.conv(x.unsqueeze(1)); B,C,F,T = h.shape; h = h.reshape(B, C*F, T)
        e = s.bn(s.proj(torch.cat([h.mean(2), h.var(2,unbiased=False).clamp_min(1e-8).sqrt()],1)))
        return e, s.head(e)

_sd = torch.load(f'{R}/code/modelA_speaker_encoder.pt', map_location=dev)
model = SpeakerNet(_sd['head.weight'].shape[0]).to(dev); model.load_state_dict(_sd); model.eval()
src = fit(load16k(f'{R}/demo_best_config/A_original.wav'))   # the clean clip played in the rooms
held = {}                                              # held-out clips, as in 09_main_run.py
for p in sorted(glob.glob(f'{R}/training_data/heldout_subset/*.flac')):
    held.setdefault(os.path.basename(p).split('_')[0], []).append(p)
held = {k: v for k, v in held.items() if len(v) >= 3}
print('rooms with recordings:', ready, '| held-out speakers:', len(held))

In [ ]:
# ===== next cell =====
def deconvolve(rec, ref, f_lo=250., f_hi=7000., lam=1e-2, tail=SR):
    n = 1 << (max(rec.shape[0], ref.shape[0])*2 - 1).bit_length()
    Y = torch.fft.rfft(rec, n); X = torch.fft.rfft(ref, n)
    f  = torch.linspace(0, SR/2, X.shape[0])
    bp = 0.5*(1+torch.tanh((f-f_lo)/(0.3*f_lo))) * 0.5*(1-torch.tanh((f-f_hi)/(0.1*f_hi)))
    reg = lam*(X.abs()**2).max() * (1 + 1e4*(1-bp))
    h = torch.fft.irfft(Y*X.conj()/(X.abs()**2 + reg) * bp, n)
    k = int(h[:SR*5].abs().argmax()); h = h[max(0,k-64) : max(0,k-64)+tail]
    h = h / h.abs().max().clamp_min(1e-9)        # <-- direct path = 1 BEFORE the blend
    Hc = torch.fft.rfft(h, tail)
    fc = torch.linspace(0, SR/2, Hc.shape[0])
    bc = 0.5*(1+torch.tanh((fc-f_lo)/(0.3*f_lo))) * 0.5*(1-torch.tanh((fc-f_hi)/(0.1*f_hi)))
    d  = torch.zeros(tail); d[64] = 1.
    h  = torch.fft.irfft(Hc*bc + torch.fft.rfft(d, tail)*(1-bc), tail)
    return h / h.abs().max().clamp_min(1e-9)

RIR_MEAS = {}
print(f'{"room":15s}{"src-real":>10s}{"src-sim":>9s}{"real-sim":>10s}{"LTAS":>7s}{"RT60":>7s}')
for r in ready:
    trimmed, _ = trim_to_sweep(load16k(f'{REC}/sweep_{r}.wav'), ref)
    h = deconvolve(trimmed, ref); RIR_MEAS[r] = h          # no truncate()
    real = align_to(load16k(f'{REC}/clean_{r}.wav'), src); sim = apply_rir(src, h)
    e0, e1, e2 = emb(model,src), emb(model,real), emb(model,sim)
    Lc = {k: (lambda v: v - v.mean())(logmel(v).mean(1)) for k,v in [('real',real),('sim',sim)]}
    print(f'{r:15s}{cos(e0,e1):10.3f}{cos(e0,e2):9.3f}{cos(e1,e2):10.3f}'
          f'{float((Lc["real"]-Lc["sim"]).abs().mean()):7.2f}{rt60(h):7.2f}')

torch.save({'rirs': RIR_MEAS, 'rooms': ready, 'held': HELD}, f'{REC}/measured_rirs.pt')

In [ ]:
# ===== ADDED for the re-run: check: the rooms rebuilt above against the rooms saved in the project folder =====
def best_xcorr(a, b, maxlag=400):                      # as in the overnight run v2
    n = 1 << (a.shape[0] + b.shape[0]).bit_length()
    c = torch.fft.irfft(torch.fft.rfft(a, n) * torch.fft.rfft(b.flip(0), n), n)
    c = c / (a.norm() * b.norm()).clamp_min(1e-12); mid = b.shape[0] - 1
    return float(c[max(0, mid - maxlag): mid + maxlag + 1].max())
_saved = torch.load(f'{ORIG}/testing_data/recordings/measured_rirs.pt')
for r in ready:
    print(f'{r:15s} RT60 {rt60(RIR_MEAS[r]):.2f} s (saved {rt60(_saved["rirs"][r].float()):.2f} s)'
          f'   similarity to the saved response {best_xcorr(RIR_MEAS[r].float(), _saved["rirs"][r].float()):.4f}')

## 08_thresholds.py

Decision thresholds measured under the test conditions (fix for mistake C2): tau_A 0.734, tau_B 0.793.

In [ ]:
# ===== ADDED for the re-run: switch encoder B to evaluation mode (the training loop in 06 leaves it in training mode) =====
modelB.eval()

In [ ]:
# ===== CELL 3a — the threshold that actually applies =====
rng = random.Random(1)
spk_list = [s for s in held if len(held[s]) >= 3][:30]
out = {}
for jn, J in [('A', model), ('B', modelB)]:
    gen, imp = [], []
    for hr in ready:
        train = [r for r in ready if r != hr]; ht = RIR_MEAS[hr]
        for s in spk_list:
            ps  = rng.sample(held[s], 2)
            enr = fit(load16k(ps[0]))
            E = nrm(torch.stack([emb(J, apply_rir(enr, RIR_MEAS[r])) for r in train]
                                + [emb(J, enr)]).mean(0))
            gen.append(cos(E, emb(J, apply_rir(fit(load16k(ps[1])), ht))))
            s2 = rng.choice([x for x in spk_list if x != s])
            imp.append(cos(E, emb(J, apply_rir(fit(load16k(rng.choice(held[s2]))), ht))))
    gen, imp = np.array(gen), np.array(imp)
    ths = np.linspace(-1, 1, 2000)
    far = np.array([(imp > t).mean() for t in ths]); frr = np.array([(gen <= t).mean() for t in ths])
    i = int(np.argmin(np.abs(far - frr))); out[jn] = float(ths[i])
    print(f'judge {jn}: EER {100*(far[i]+frr[i])/2:5.2f}%   tau {ths[i]:.3f}')
    print(f'   genuine  {gen.mean():.3f} +- {gen.std():.3f}'
          f'    impostor {imp.mean():.3f} +- {imp.std():.3f}')
TAU_A, TAU_B = out['A'], out['B']